# Claim2Value：在云端分割训练文件并写回 Drive

在 **Google Colab** 中选择「运行时 → 全部运行」。所有数据读取、分割与上传都发生在 Google 云端，不需要把 623 MB 文件下载到你的电脑。

若 Google 提示登录/授权，请本人选择有赛事训练文件访问权的 Google 账号。Colab 的凭据与 ChatGPT 的 Drive 插件连接独立；我不能从插件取出或复用账号令牌。

此笔记本已做格式、语法和本地模拟接口验证，**尚未在你的 Colab 账号中执行**。成功标志是最后输出 `COMPLETE`，以及新文件夹内三个 part 文件和一个 manifest。


## 1. 连接自己的 Google 账号

使用 Google 官方 Colab 认证与 Drive API。凭据只保留在当前 Colab 运行环境中，不写入 notebook 输出。依赖通常已安装；缺失时在这个云端运行环境安装 SDK。


In [ ]:
import sys, subprocess
try:
    import googleapiclient
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'google-api-python-client'], check=True)

from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
print('Google 账号已连接；下一步只处理指定训练文件和项目输出目录。')


## 2. 加载已检查的传输代码

代码完整嵌入本 notebook，运行时不从 GitHub 拉取可变源码。官方训练文件 ID 和输出父目录已填好；原始文件内容及共享权限保持原样。


In [ ]:
import types
byte_transfer = types.ModuleType('claim2value_byte_transfer')
BYTE_SOURCE = r'''"""Split and reassemble official training bytes without changing the Parquet file."""
from __future__ import annotations
import argparse
import hashlib
import json
from pathlib import Path

TRAIN_SHA = "482d4c0387e9da2facd77f722fa465b2d69ec7f119d96933c57761790f4d9519"
PART_BYTES = 200 * 1024 * 1024
BUFFER_BYTES = 8 * 1024 * 1024


def digest(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(BUFFER_BYTES), b""):
            result.update(block)
    return result.hexdigest()


def split(source, out, expected_sha=TRAIN_SHA, part_bytes=PART_BYTES):
    source, out = Path(source), Path(out)
    if not isinstance(part_bytes, int) or not 0 < part_bytes <= PART_BYTES:
        raise ValueError("Part size must be positive and at most 200 MiB")
    if digest(source) != expected_sha:
        raise ValueError("Source SHA-256 differs from the expected original training file")
    if not source.stat().st_size:
        raise ValueError("Empty source")
    out.mkdir(parents=True, exist_ok=False)
    parts, total = [], hashlib.sha256()
    with source.open("rb") as handle:
        while block := handle.read(min(BUFFER_BYTES, part_bytes)):
            name = f"train.parquet.part{len(parts) + 1:03d}"
            current, written = hashlib.sha256(), 0
            with (out / name).open("xb") as destination:
                while block:
                    destination.write(block)
                    current.update(block)
                    total.update(block)
                    written += len(block)
                    if written == part_bytes:
                        break
                    block = handle.read(min(BUFFER_BYTES, part_bytes - written))
            parts.append({"file": name, "bytes": written, "sha256": current.hexdigest()})
    if total.hexdigest() != expected_sha:
        raise ValueError("Source changed during splitting; no completed manifest created")
    manifest = {"format": "original_byte_parts_v1", "original_name": "train.parquet",
                "original_bytes": sum(p["bytes"] for p in parts), "original_sha256": expected_sha,
                "part_bytes": part_bytes, "parts": parts}
    with (out / "parts_manifest.json").open("x", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)
    return manifest


def join(parts_dir, output, expected_sha=TRAIN_SHA):
    parts_dir, output = Path(parts_dir), Path(output)
    manifest = json.loads((parts_dir / "parts_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("format") != "original_byte_parts_v1" or manifest.get("original_sha256") != expected_sha:
        raise ValueError("Training manifest identity mismatch")
    parts = manifest["parts"]
    names = [p["file"] for p in parts]
    if not names or names != [f"train.parquet.part{i + 1:03d}" for i in range(len(parts))]:
        raise ValueError("Missing, reordered or unsafe part names")
    for part in parts:
        path = parts_dir / part["file"]
        if not 0 < part["bytes"] <= PART_BYTES or path.stat().st_size != part["bytes"] or digest(path) != part["sha256"]:
            raise ValueError("Part size or SHA-256 mismatch")
    if sum(p["bytes"] for p in parts) != manifest["original_bytes"]:
        raise ValueError("Manifest total size mismatch")
    output.parent.mkdir(parents=True, exist_ok=True)
    result = hashlib.sha256()
    with output.open("xb") as destination:
        for part in parts:
            with (parts_dir / part["file"]).open("rb") as source:
                for block in iter(lambda: source.read(BUFFER_BYTES), b""):
                    destination.write(block)
                    result.update(block)
    if result.hexdigest() != expected_sha:
        output.unlink()
        raise ValueError("Reassembled file differs from official original; output removed")
    return {"status": "REASSEMBLED", "bytes": output.stat().st_size, "sha256": result.hexdigest()}


def main():
    p = argparse.ArgumentParser(description=__doc__)
    commands = p.add_subparsers(dest="command", required=True)
    splitter = commands.add_parser("split")
    splitter.add_argument("--input", type=Path, required=True)
    splitter.add_argument("--out", type=Path, required=True)
    assembler = commands.add_parser("join")
    assembler.add_argument("--parts", type=Path, required=True)
    assembler.add_argument("--out", type=Path, required=True)
    a = p.parse_args()
    value = split(a.input, a.out) if a.command == "split" else join(a.parts, a.out)
    print(json.dumps(value, indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(BYTE_SOURCE, 'byte_transfer.py', 'exec'), byte_transfer.__dict__)
cloud_transfer = types.ModuleType('claim2value_cloud_transfer')
CLOUD_SOURCE = r'''"""Colab SDK transfer: private Drive source -> exact byte parts -> private Drive."""
from __future__ import annotations
import datetime as dt
import hashlib
import json
from pathlib import Path
import tempfile
import uuid

TRAIN_ID = "11AN832isGfPMYirAT37VJBhmNCnNZqFw"
PARENT_ID = "1kkaTaXQg4q8B8THVRY-sUMp0Dd7vvtdx"
TRAIN_SHA = "482d4c0387e9da2facd77f722fa465b2d69ec7f119d96933c57761790f4d9519"
TRAIN_BYTES = 622965149
CHUNK_BYTES = 8 * 1024 * 1024


def hash_file(path, algorithm):
    value = hashlib.new(algorithm)
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(CHUNK_BYTES), b""):
            value.update(block)
    return value.hexdigest()


def run_transfer(service, *, split_function=None, work_parent=None,
                 downloader_factory=None, uploader_factory=None,
                 train_id=TRAIN_ID, parent_id=PARENT_ID,
                 expected_sha=TRAIN_SHA, expected_bytes=TRAIN_BYTES,
                 part_bytes=200 * 1024 * 1024):
    if split_function is None:
        from scripts.transfer_training_data import split
        split_function = split
    if downloader_factory is None or uploader_factory is None:
        from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
        downloader_factory = downloader_factory or MediaIoBaseDownload
        uploader_factory = uploader_factory or MediaFileUpload
    metadata = service.files().get(fileId=train_id,
        fields="id,name,size,mimeType,capabilities(canDownload)").execute()
    if int(metadata.get("size", -1)) != expected_bytes or not metadata.get("capabilities", {}).get("canDownload"):
        raise ValueError("Original training file size or download permission differs")
    parent = service.files().get(fileId=parent_id, fields="id,mimeType,capabilities(canAddChildren)").execute()
    if parent.get("mimeType") != "application/vnd.google-apps.folder" or not parent.get("capabilities", {}).get("canAddChildren"):
        raise ValueError("Destination is not a writable Drive folder")
    work = Path(tempfile.mkdtemp(prefix="claim2value_cloud_transfer_", dir=work_parent))
    source = work / "train.parquet"
    print("Downloading into the Colab runtime; no download to your computer", flush=True)
    request = service.files().get_media(fileId=train_id)
    with source.open("xb") as handle:
        downloader = downloader_factory(handle, request, chunksize=CHUNK_BYTES)
        done, previous = False, -1
        while not done:
            status, done = downloader.next_chunk(num_retries=3)
            if status:
                percent = int(status.progress() * 100)
                if percent // 10 != previous:
                    previous = percent // 10
                    print(f"Download {percent}%", flush=True)
    if source.stat().st_size != expected_bytes or hash_file(source, "sha256") != expected_sha:
        raise ValueError("Downloaded bytes differ from the official original; nothing written to Drive")
    parts_dir = work / "parts"
    manifest = split_function(source, parts_dir, expected_sha=expected_sha, part_bytes=part_bytes)
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    name = f"Claim2Value_train_parts_{stamp}_{uuid.uuid4().hex[:8]}"
    folder = service.files().create(body={"name": name, "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id]}, fields="id,name,webViewLink").execute()

    def upload(path, mime_type):
        request = service.files().create(body={"name": path.name, "parents": [folder["id"]]},
            media_body=uploader_factory(str(path), mimetype=mime_type, chunksize=CHUNK_BYTES, resumable=True),
            fields="id,name,size,md5Checksum")
        response = None
        while response is None:
            _, response = request.next_chunk(num_retries=3)
        if int(response.get("size", -1)) != path.stat().st_size or response.get("md5Checksum") != hash_file(path, "md5"):
            raise ValueError("Uploaded file byte count/checksum mismatch; transfer remains incomplete")
        return response

    for number, part in enumerate(manifest["parts"], 1):
        response = upload(parts_dir / part["file"], "application/octet-stream")
        part["drive_file_id"] = response["id"]
        print(f"Verified part {number}/{len(manifest['parts'])}", flush=True)
    # The completed manifest appears only after all parts have been uploaded and checked.
    manifest["source_drive_file_id"] = train_id
    manifest["destination_drive_folder_id"] = folder["id"]
    manifest["transfer_scope"] = "ORIGINAL_BYTES_ONLY_NO_MODELING"
    manifest_path = parts_dir / "parts_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    manifest_file = upload(manifest_path, "application/json")
    result = {"status": "COMPLETE", "folder_id": folder["id"], "folder_name": folder["name"],
              "folder_url": folder.get("webViewLink"), "manifest_file_id": manifest_file["id"],
              "parts": len(manifest["parts"]), "original_sha256": expected_sha,
              "raw_original_modified": False, "model_training_run": False}
    print(json.dumps(result, ensure_ascii=False, indent=2), flush=True)
    return result
'''
exec(compile(CLOUD_SOURCE, 'cloud_transfer.py', 'exec'), cloud_transfer.__dict__)
print('原始字节传输工具已加载。')


## 3. 在云端完成传输

先按 8 MiB 请求将原始数据读入 Colab 临时磁盘，核对官方 SHA-256，再按最多 200 MiB 分为三段，并在已有项目目录下创建新子文件夹。每段上传后检查服务器字节数与 MD5；全部成功才上传最终 manifest。

中断或错误不会标为完成，已生成的新文件夹保留现场。重复执行会另建文件夹；原文件不会被覆盖。这里不运行建模、不读取测试收益。


In [ ]:
result = cloud_transfer.run_transfer(service, split_function=byte_transfer.split)
assert result['status'] == 'COMPLETE'


## 4. 成功后

看到 `COMPLETE` 后，把输出文件夹链接或名字发给助手。助手可通过已有 Drive 连接下载三段、拼回原文件，再核对完整官方指纹并执行既定技术题流程。若报错，把错误文字回传；`COMPLETE` 尚未出现的目录不能当成完整传输。

Google 官方参考：

- [Drive 下载与按字节读取](https://developers.google.com/workspace/drive/api/guides/manage-downloads)
- [Colab 外部数据与认证](https://colab.research.google.com/notebooks/io.ipynb)
